In [1]:
import os
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel, TrainingArguments, Trainer
from datasets import Dataset


tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2", device_map='auto')
tokenizer.pad_token = tokenizer.eos_token


data = []
with open(os.path.join("E2E", "train.txt"), 'r', encoding='utf8') as f:
	for line in f:
		context, completion = line.strip().split('||')
		data.append({'context': context, 'completion': completion})
train_dataset = Dataset.from_list(data)

data = []
with open(os.path.join("E2E", "valid.txt"), 'r', encoding='utf8') as f:
	for line in f:
		context, completion = line.strip().split('||')
		data.append({'context': context, 'completion': completion})
valid_dataset = Dataset.from_list(data)

data = []
with open(os.path.join("E2E", "test.txt"), 'r', encoding='utf8') as f:
	for line in f:
		context, completion = line.strip().split('||')
		data.append({'context': context, 'completion': completion})
test_dataset = Dataset.from_list(data)

c:\Users\3un8i\anaconda3\envs\IDA\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
max_seq_len = 144

tokenized_data = []
for example in train_dataset:
    tokenized_context = tokenizer.encode(example['context'] + tokenizer.bos_token)
    tokenized_completion = tokenizer.encode(' ' + example['completion'] + tokenizer.eos_token)

    input_ids = tokenized_context + tokenized_completion + [tokenizer.pad_token_id for _ in range(max_seq_len - len(tokenized_context) - len(tokenized_completion))]
    attention_mask = [1 for _ in range(len(tokenized_context) + len(tokenized_completion))] + [0 for _ in range(max_seq_len - len(tokenized_context) - len(tokenized_completion))]
    labels = [-100 for _ in tokenized_context] + tokenized_completion + [-100 for _ in range(max_seq_len - len(tokenized_context) - len(tokenized_completion))]
    tokenized_data.append({'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': labels})
preprocessed_train_dataset = Dataset.from_list(tokenized_data)

tokenized_data = []
for example in valid_dataset:
    tokenized_context = tokenizer.encode(example['context'] + tokenizer.bos_token)
    tokenized_completion = tokenizer.encode(' ' + example['completion'] + tokenizer.eos_token)

    input_ids = tokenized_context + tokenized_completion + [tokenizer.pad_token_id for _ in range(max_seq_len - len(tokenized_context) - len(tokenized_completion))]
    attention_mask = [1 for _ in range(len(tokenized_context) + len(tokenized_completion))] + [0 for _ in range(max_seq_len - len(tokenized_context) - len(tokenized_completion))]
    labels = [-100 for _ in tokenized_context] + tokenized_completion + [-100 for _ in range(max_seq_len - len(tokenized_context) - len(tokenized_completion))]
    tokenized_data.append({'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': labels})
preprocessed_valid_dataset = Dataset.from_list(tokenized_data)

tokenized_data = []
for example in test_dataset:
    tokenized_context = tokenizer.encode(example['context'] + tokenizer.bos_token)
    tokenized_completion = tokenizer.encode(' ' + example['completion'] + tokenizer.eos_token)

    input_ids = tokenized_context + tokenized_completion + [tokenizer.pad_token_id for _ in range(max_seq_len - len(tokenized_context) - len(tokenized_completion))]
    attention_mask = [1 for _ in range(len(tokenized_context) + len(tokenized_completion))] + [0 for _ in range(max_seq_len - len(tokenized_context) - len(tokenized_completion))]
    labels = [-100 for _ in tokenized_context] + tokenized_completion + [-100 for _ in range(max_seq_len - len(tokenized_context) - len(tokenized_completion))]
    tokenized_data.append({'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': labels})
preprocessed_test_dataset = Dataset.from_list(tokenized_data)

In [3]:
training_args = TrainingArguments(
    output_dir="./gpt2_e2e_model",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=8,
    learning_rate=8e-5,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="epoch",
    save_steps=1,
    save_total_limit=5
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=preprocessed_train_dataset,
    eval_dataset=preprocessed_valid_dataset
)

trainer.train()

  1%|▏         | 100/7887 [00:38<48:38,  2.67it/s]

{'loss': 3.0672, 'grad_norm': 19.646316528320312, 'learning_rate': 1.6000000000000003e-05, 'epoch': 0.04}


  3%|▎         | 200/7887 [01:15<47:38,  2.69it/s]

{'loss': 1.613, 'grad_norm': 22.56386375427246, 'learning_rate': 3.2000000000000005e-05, 'epoch': 0.08}


  4%|▍         | 300/7887 [01:53<47:29,  2.66it/s]

...

 99%|█████████▉| 7800/7887 [55:46<00:32,  2.69it/s]

{'loss': 1.0141, 'grad_norm': 5.26118278503418, 'learning_rate': 9.42195749289292e-07, 'epoch': 2.97}


100%|██████████| 7887/7887 [56:22<00:00,  2.33it/s]

{'train_runtime': 3382.6502, 'train_samples_per_second': 37.303, 'train_steps_per_second': 2.332, 'train_loss': 1.1321685972875162, 'epoch': 3.0}


TrainOutput(global_step=7887, training_loss=1.1321685972875162, metrics={'train_runtime': 3382.6502, 'train_samples_per_second': 37.303, 'train_steps_per_second': 2.332, 'total_flos': 9272984758272000.0, 'train_loss': 1.1321685972875162, 'epoch': 3.0})

In [4]:
list_test = []

for i in range(len(test_dataset)):
    if len(list_test) == 0 or list_test[-1]['context'] != test_dataset[i]['context']:
        list_test.append({'context': test_dataset[i]['context'], 'references': [test_dataset[i]['completion']]})
    else:
        list_test[-1]['references'].append(test_dataset[i]['completion'])

print(len(list_test))

630


In [5]:
model.eval()

for i in range(len(list_test)):
    encoded = tokenizer(list_test[i]['context'] + tokenizer.bos_token, return_tensors='pt').to('cuda')
    output = model.generate(
        input_ids=encoded['input_ids'],
        attention_mask=encoded['attention_mask'],
        max_length=max_seq_len,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )
    output = tokenizer.decode(output[0])
    list_test[i]['prediction'] = output[len(list_test[i]['context']):]

for i in range(len(list_test)):
    print(list_test[i]['prediction'])

<|endoftext|> Blue Spice is a coffee shop in the city centre .<|endoftext|>
<|endoftext|> Blue Spice is a coffee shop in the riverside area .<|endoftext|>
<|endoftext|> Blue Spice is a coffee shop near Crowne Plaza Hotel . It has a customer rating of 5 out of 5 .<|endoftext|>
<|endoftext|> Blue Spice is a coffee shop near Burger King with an average customer rating .<|endoftext|>
<|endoftext|> Blue Spice is a coffee shop near Crowne Plaza Hotel . It has an average customer rating .<|endoftext|>
<|endoftext|> Blue Spice is a pub in the city centre .<|endoftext|>
<|endoftext|> Blue Spice is a pub in the riverside area .<|endoftext|>
<|endoftext|> Blue Spice is a pub near Crowne Plaza Hotel with a customer rating of 5 out of 5 .<|endoftext|>
<|endoftext|> Blue Spice is a pub near Burger King with an average customer rating .<|endoftext|>
<|endoftext|> Blue Spice is a pub near Crowne Plaza Hotel with an average customer rating .<|endoftext|>
<|endoftext|> Blue Spice is a pub that serves Ch

In [6]:
list_references = []
list_prediction = []

for i in range(len(list_test)):
    list_references.append(list_test[i]['references'])
    list_prediction.append(list_test[i]['prediction'][len(tokenizer.bos_token)+1:-len(tokenizer.eos_token)])

print(len(list_references))
print(len(list_prediction))

630
630


In [7]:
def save_benchmark_data(list_references, list_prediction, reference_file, prediction_file):
    with open(reference_file, "w", encoding="utf-8") as ref_file:
        for refs in list_references:
            for ref in refs:
                ref_file.write(ref + "\n")
            ref_file.write("\n")

    with open(prediction_file, "w", encoding="utf-8") as pred_file:
        for pred in list_prediction:
            pred_file.write(pred + "\n")

reference_file = "references.txt"
prediction_file = "predictions.txt"

save_benchmark_data(list_references, list_prediction, reference_file, prediction_file)